# Analyse sur le taux de completion faible des contributions (via BDD)

On va se concentrer sur la contribution les-conges-pour-evenements-familiaux qui est la plus visitée

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, timedelta
from analysis.connectors.matomo import MatomoSQLConnector

matomo = MatomoSQLConnector()
await matomo.connect()

1. Chargement des données

On charge juin 2026 en filtrant côté SQL sur ce qui concerne les contributions : les page_views dont l'URL contient /contribution/ et les events de catégorie contribution. Cela réduit fortement le volume sans changer la logique.

In [ ]:
interval_start = '2026-08-01 00:00:00'
interval_stop = '2026-09-01 00:00:00'

In [ ]:
columns = ['action_id', 'idvisit', 'actions', 'action_type',
           'action_eventcategory', 'action_eventaction', 'action_eventname',
           'action_eventvalue', 'action_url', 'action_timestamp', 'experiments',
           'referrertype', 'referrername', 'visitduration']

query_visits = f"""
    SELECT {", ".join(columns)} FROM matomo_partitioned
    WHERE action_timestamp >= '{interval_start}'
      AND action_timestamp < '{interval_stop}'
    ORDER BY action_timestamp asc;
"""
visits_data = await matomo.run_query(query_visits)
visits_df = pd.DataFrame(visits_data, columns=columns)

On récupère les visites où l'utilisateur a visité la contribution

In [ ]:
from urllib.parse import urlsplit


TARGET_PATH = '/contribution/quel-est-le-salaire-minimum-dun-alternant-en-2026'

# Normalisation d'URL : on retire query string (?src_url=…), ancre et / final
def norm_path(u):
    if not isinstance(u, str):
        return None
    return urlsplit(u).path.rstrip('/') or '/'

visits_df['path'] = visits_df['action_url'].map(norm_path)
visits_df['actions'] = pd.to_numeric(visits_df['actions'], errors='coerce')

# Visites ayant vu la page cible (page_view, chemin exact, hors widgets)
mask_page = (visits_df['action_type'] == 'action') & (visits_df['path'] == TARGET_PATH)
target_visits = visits_df.loc[mask_page, 'idvisit'].unique()

df = visits_df[visits_df['idvisit'].isin(target_visits)].sort_values(['idvisit', 'actions'])
print(f"{len(target_visits)} visites sur {TARGET_PATH} en août")

Le but est de confirmer les noms d'events avant de définir « complété » et les segments de sortie. On restreint aux lignes dont l'URL est la page cible, ou dont l'action_eventname contient le slug (cas des events contribution et cc_search_type_of_users qui embarquent le path).

In [ ]:
on_page = df[
    (df['action_type'] == 'event') &
    ((df['path'] == TARGET_PATH) | df['action_eventname'].str.contains(TARGET_PATH, na=False))
]

inventory = (on_page
    .groupby(['action_eventcategory', 'action_eventaction'])
    .agg(n_events=('action_id', 'size'),
         n_visits=('idvisit', 'nunique'),
         exemple_name=('action_eventname', 'first'))
    .sort_values('n_visits', ascending=False))

inventory['pct_visits'] = (inventory['n_visits'] / len(target_visits) * 100).round(1)
inventory

## exclusion des visites ayant consulté le contenu

In [ ]:
COMPLETION_ACTIONS = {
    'click_afficher_les_informations_CC',
    'click_afficher_les_informations_sans_CC',
    'click_afficher_les_informations_générales',
}

is_event = df['action_type'] == 'event'
on_target = (df['path'] == TARGET_PATH) | df['action_eventname'].str.contains(TARGET_PATH, na=False)

mask_afficher = (df['action_eventcategory'] == 'contribution') & df['action_eventaction'].isin(COMPLETION_ACTIONS)
mask_pe = (df['action_eventcategory'] == 'cc_search_type_of_users') & (df['action_eventaction'] == 'select_je_n_ai_pas_d_entreprise')

mask_completion = is_event & on_target & (mask_afficher | mask_pe)

completed_visits = set(df.loc[mask_completion, 'idvisit'].unique())
not_completed_visits = set(target_visits) - completed_visits

n_total = len(target_visits)
print(f"Visites sur la page      : {n_total}")
print(f"Contenu consulté         : {len(completed_visits)} ({len(completed_visits)/n_total:.1%})")
print(f"Contenu non consulté     : {len(not_completed_visits)} ({len(not_completed_visits)/n_total:.1%})")

# Répartition par voie de complétion (une visite peut en avoir plusieurs)
(df.loc[mask_completion]
   .groupby(['action_eventcategory', 'action_eventaction'])['idvisit'].nunique()
   .sort_values(ascending=False))

df_nc = df[df['idvisit'].isin(not_completed_visits)].copy()

## sortie du site vs pages consultées ensuite

In [ ]:
df_nc = df_nc.sort_values(['idvisit', 'actions'])

# Ordre (colonne actions) de la dernière page vue de la page cible, par visite
last_target = (df_nc[(df_nc['action_type'] == 'action') & (df_nc['path'] == TARGET_PATH)]
               .groupby('idvisit')['actions'].max()
               .rename('last_target_action'))

df_nc = df_nc.join(last_target, on='idvisit')

# Pages vues après cette dernière occurrence
after = df_nc[(df_nc['action_type'] == 'action') &
              (df_nc['actions'] > df_nc['last_target_action'])]

visits_with_next = set(after['idvisit'].unique())
visits_exit = set(not_completed_visits) - visits_with_next

n_nc = len(not_completed_visits)
print(f"Visites non complétées           : {n_nc}")
print(f"Quittent le site après la page   : {len(visits_exit)} ({len(visits_exit)/n_nc:.1%})")
print(f"Continuent sur le site           : {len(visits_with_next)} ({len(visits_with_next)/n_nc:.1%})")

In [ ]:
# Page immédiatement suivante
next_page = (after.sort_values(['idvisit', 'actions'])
                  .groupby('idvisit').first()['path'])

next_page_stats = (next_page.value_counts()
                   .rename('n_visits').to_frame())
next_page_stats['pct_continuent'] = (next_page_stats['n_visits'] / len(visits_with_next) * 100).round(1)
next_page_stats.head(30)

## profil des visites qui quittent le site

In [ ]:
df_exit = df_nc[df_nc['idvisit'].isin(visits_exit)].copy()

# Attributs de visite (première ligne de chaque visite)
visit_attrs = (df_exit.sort_values(['idvisit', 'actions'])
                      .groupby('idvisit')
                      .agg(referrertype=('referrertype', 'first'),
                           referrername=('referrername', 'first'),
                           visitduration=('visitduration', 'first')))
visit_attrs['visitduration'] = pd.to_numeric(visit_attrs['visitduration'], errors='coerce')

# Nombre de pages vues et rang de la page cible dans la visite
pages = df_exit[df_exit['action_type'] == 'action'].sort_values(['idvisit', 'actions'])
pages['rank'] = pages.groupby('idvisit').cumcount() + 1
visit_attrs['n_pages'] = pages.groupby('idvisit').size()
visit_attrs['target_rank'] = (pages[pages['path'] == TARGET_PATH]
                              .groupby('idvisit')['rank'].min())
visit_attrs['landing_on_target'] = visit_attrs['target_rank'] == 1

n_exit = len(visit_attrs)
print(f"Visites sortantes : {n_exit}")
print(f"Arrivées directement sur la page cible : {visit_attrs['landing_on_target'].mean():.1%}")

In [ ]:
ref_type = visit_attrs['referrertype'].value_counts(dropna=False).rename('n_visits').to_frame()
ref_type['pct'] = (ref_type['n_visits'] / n_exit * 100).round(1)
ref_type

In [ ]:
ref_name = visit_attrs['referrername'].value_counts(dropna=False).head(20).rename('n_visits').to_frame()
ref_name['pct'] = (ref_name['n_visits'] / n_exit * 100).round(1)
ref_name

In [ ]:
pages_dist = visit_attrs['n_pages'].clip(upper=10).value_counts().sort_index().rename('n_visits').to_frame()
pages_dist['pct'] = (pages_dist['n_visits'] / n_exit * 100).round(1)
pages_dist  # la ligne 10 = "10 pages et plus"

In [ ]:
bins = [-1, 0, 10, 30, 60, 120, 300, 600, float('inf')]
labels = ['0s', '1-10s', '11-30s', '31-60s', '1-2min', '2-5min', '5-10min', '>10min']
visit_attrs['duration_bucket'] = pd.cut(visit_attrs['visitduration'], bins=bins, labels=labels)
dur_dist = visit_attrs['duration_bucket'].value_counts(sort=False).rename('n_visits').to_frame()
dur_dist['pct'] = (dur_dist['n_visits'] / n_exit * 100).round(1)
dur_dist

In [ ]:
pd.crosstab(visit_attrs['landing_on_target'], visit_attrs['duration_bucket'], normalize='index').round(3) * 100

In [ ]:
events_exit = df_exit[
    (df_exit['action_type'] == 'event') &
    ((df_exit['path'] == TARGET_PATH) | df_exit['action_eventname'].str.contains(TARGET_PATH, na=False))
]

visit_attrs['n_events_page'] = events_exit.groupby('idvisit').size()
visit_attrs['n_events_page'] = visit_attrs['n_events_page'].fillna(0).astype(int)

n_events_dist = visit_attrs['n_events_page'].clip(upper=10).value_counts().sort_index().rename('n_visits').to_frame()
n_events_dist['pct'] = (n_events_dist['n_visits'] / n_exit * 100).round(1)
n_events_dist  # la ligne 10 = "10 events et plus"

In [ ]:
ev_inventory = (events_exit
    .groupby(['action_eventcategory', 'action_eventaction'])
    .agg(n_events=('action_id', 'size'), n_visits=('idvisit', 'nunique'))
    .sort_values('n_visits', ascending=False))
ev_inventory['pct_visits_sortantes'] = (ev_inventory['n_visits'] / n_exit * 100).round(1)
ev_inventory

In [ ]:
visit_attrs['has_events'] = visit_attrs['n_events_page'] > 0
pd.crosstab([visit_attrs['landing_on_target'], visit_attrs['has_events']],
            visit_attrs['duration_bucket'])

In [ ]:
mask_p1 = (
    (df_nc['action_type'] == 'event') &
    (df_nc['action_eventcategory'] == 'cc_search_type_of_users') &
    (df_nc['action_eventaction'] == 'click_p1') &
    ((df_nc['path'] == TARGET_PATH) | df_nc['action_eventname'].str.contains(TARGET_PATH, na=False))
)
visits_p1 = df_nc.loc[mask_p1, 'idvisit'].unique()
print(f"{len(visits_p1)} visites non complétées avec click_p1 sur la page")

def show_visit(idvisit, frame=df_nc):
    v = frame[frame['idvisit'] == idvisit].sort_values('actions')
    cols = ['actions', 'action_timestamp', 'action_type', 'path',
            'action_eventcategory', 'action_eventaction', 'action_eventname', 'action_eventvalue']
    with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 80, 'display.width', 250):
        print(f"idvisit={idvisit} | durée={v['visitduration'].iloc[0]}s | "
              f"referrer={v['referrertype'].iloc[0]} / {v['referrername'].iloc[0]}")
        display(v[cols].reset_index(drop=True))

show_visit(visits_p1[0])

In [ ]:
on_target_nc = (df_nc['path'] == TARGET_PATH) | df_nc['action_eventname'].str.contains(TARGET_PATH, na=False)
is_event_nc = df_nc['action_type'] == 'event'

# Visites non complétées ayant engagé la recherche de CC sur la page
visits_search = set(df_nc.loc[is_event_nc & on_target_nc &
                              (df_nc['action_eventcategory'] == 'cc_search_type_of_users'), 'idvisit'])

# ... dont CC sélectionnée non traitée
visits_search_nt = set(df_nc.loc[is_event_nc & on_target_nc &
                                 (df_nc['action_eventcategory'] == 'outil') &
                                 (df_nc['action_eventaction'] == 'cc_select_non_traitée'), 'idvisit']) & visits_search

n_nc = len(not_completed_visits)
print(f"Visites non complétées                                  : {n_nc}")
print(f"  dont recherche de CC engagée (cc_search_type_of_users) : {len(visits_search)} ({len(visits_search)/n_nc:.1%})")
print(f"  dont CC non traitée sélectionnée                       : {len(visits_search_nt)} "
      f"({len(visits_search_nt)/len(visits_search):.1%} des abandons avec recherche, "
      f"{len(visits_search_nt)/n_nc:.1%} de tous les abandons)")

In [ ]:
def visits_with(cat, act=None):
    m = is_event_nc & on_target_nc & (df_nc['action_eventcategory'] == cat)
    if act:
        m &= df_nc['action_eventaction'] == act
    return set(df_nc.loc[m, 'idvisit']) & visits_search

steps = {
    'CC non traitée sélectionnée': visits_search_nt,
    'CC traitée sélectionnée (sans clic afficher)': visits_with('outil', 'cc_select_traitée'),
    'CC sélectionnée P1/P2 sans event outil': visits_with('cc_select_p1') | visits_with('cc_select_p2'),
    'Recherche entreprise sans sélection de CC': visits_with('enterprise_search'),
}
remaining = set(visits_search)
rows = []
for label, s in steps.items():
    s = s & remaining
    rows.append((label, len(s)))
    remaining -= s
rows.append(('Choix de parcours (click_p*) sans suite', len(remaining)))

pd.DataFrame(rows, columns=['étape atteinte', 'n_visits']).assign(
    pct=lambda d: (d['n_visits'] / len(visits_search) * 100).round(1))

## Rattachement des events à leur page courante (à calculer une fois sur df_nc, réutilisable ensuite)

In [ ]:
df_nc = df_nc.sort_values(['idvisit', 'actions']).copy()

# Page courante = path du dernier page_view précédent dans la visite (forward fill par visite)
df_nc['current_page'] = df_nc['path'].where(df_nc['action_type'] == 'action')
df_nc['current_page'] = df_nc.groupby('idvisit')['current_page'].ffill()

In [ ]:
df_exit = df_nc[df_nc['idvisit'].isin(visits_exit)].copy()

ALLOWED_EVENTS = {
    ('enterprise_search',       TARGET_PATH),
    ('enterprise_select',       TARGET_PATH),
    ('cc_select_p1',            TARGET_PATH),
    ('cc_select_p2',            TARGET_PATH),
    ('outil',                   'cc_select_traitée'),
    ('outil',                   'cc_select_non_traitée'),
    ('cc_search_type_of_users', 'click_p1'),
    ('cc_search_type_of_users', 'click_p2'),
    ('cc_search_type_of_users', 'click_p3'),
}
pairs = list(zip(df_exit['action_eventcategory'], df_exit['action_eventaction']))
is_allowed = pd.Series([p in ALLOWED_EVENTS for p in pairs], index=df_exit.index)

events_exit = df_exit[(df_exit['action_type'] == 'event') &
                      (df_exit['current_page'] == TARGET_PATH) &
                      is_allowed]

visit_attrs['n_events_page'] = events_exit.groupby('idvisit').size()
visit_attrs['n_events_page'] = visit_attrs['n_events_page'].fillna(0).astype(int)

n_events_dist = visit_attrs['n_events_page'].clip(upper=10).value_counts().sort_index().rename('n_visits').to_frame()
n_events_dist['pct'] = (n_events_dist['n_visits'] / n_exit * 100).round(1)
n_events_dist  # la ligne 10 = "10 events et plus"

## Combien d'utilisateur sont arrivé globalement directement sur le site

## On se concentre sur les visites non complété où l'utilisateur est arrivé directement sur la page

In [ ]:
pages_nc = df_nc[df_nc['action_type'] == 'action'].sort_values(['idvisit', 'actions'])
first_page = pages_nc.groupby('idvisit')['path'].first()

visits_landing = set(first_page[first_page == TARGET_PATH].index)
df_landing = df_nc[df_nc['idvisit'].isin(visits_landing)]

n_nc = len(not_completed_visits)
n_landing = len(visits_landing)
print(f"Visites non complétées                        : {n_nc}")
print(f"  arrivées directement sur la contribution    : {n_landing} ({n_landing/n_nc:.1%})")

In [ ]:
landing_attrs = (df_landing.sort_values(['idvisit', 'actions'])
                           .groupby('idvisit')
                           .agg(referrertype=('referrertype', 'first'),
                                referrername=('referrername', 'first')))

ref_type = landing_attrs['referrertype'].value_counts(dropna=False).rename('n_visits').to_frame()
ref_type['pct'] = (ref_type['n_visits'] / n_landing * 100).round(1)
ref_type

In [ ]:
ref_name = (landing_attrs.groupby(['referrertype', 'referrername'], dropna=False)
                         .size().sort_values(ascending=False)
                         .rename('n_visits').to_frame())
ref_name['pct'] = (ref_name['n_visits'] / n_landing * 100).round(1)
ref_name.head(25)

## les chiffres d'interaction avec la modale pour les usagers qui arrivent sur une contribution personnalisée depuis un moteur de recherche ? 

In [ ]:
pages_all = df[df['action_type'] == 'action'].sort_values(['idvisit', 'actions'])
first_page_all = pages_all.groupby('idvisit')['path'].first()

visits_landing_all = set(first_page_all[first_page_all == TARGET_PATH].index)
df_landing_all = df[df['idvisit'].isin(visits_landing_all)]

n_total = len(target_visits)
n_landing_all = len(visits_landing_all)
print(f"Visites sur la page                          : {n_total}")
print(f"  ayant commencé par la contribution         : {n_landing_all} ({n_landing_all/n_total:.1%})")

In [ ]:
landing_flag = first_page_all.eq(TARGET_PATH).rename('landing_on_target')
completed_flag = pd.Series(first_page_all.index.isin(completed_visits), index=first_page_all.index, name='completed')

tab = pd.crosstab(landing_flag, completed_flag)
tab['n_visits'] = tab.sum(axis=1)
tab['taux_completion'] = (tab[True] / tab['n_visits'] * 100).round(1)
tab

In [ ]:
landing_attrs_all = (df_landing_all.sort_values(['idvisit', 'actions'])
                                   .groupby('idvisit')
                                   .agg(referrertype=('referrertype', 'first'),
                                        referrername=('referrername', 'first')))
landing_attrs_all['completed'] = landing_attrs_all.index.isin(completed_visits)

by_ref = (landing_attrs_all.groupby('referrertype', dropna=False)
                           .agg(n_visits=('completed', 'size'), n_completed=('completed', 'sum')))
by_ref['pct_visits'] = (by_ref['n_visits'] / n_landing_all * 100).round(1)
by_ref['taux_completion'] = (by_ref['n_completed'] / by_ref['n_visits'] * 100).round(1)
by_ref.sort_values('n_visits', ascending=False)

In [ ]:
visits_search_landing = set(landing_attrs_all[landing_attrs_all['referrertype'] == 'search'].index)

df_search = (df[df['idvisit'].isin(visits_search_landing)]
             .sort_values(['idvisit', 'actions'])
             .copy())

# Page courante pour chaque ligne (utile pour scoper les events à la contribution)
df_search['current_page'] = df_search['path'].where(df_search['action_type'] == 'action')
df_search['current_page'] = df_search.groupby('idvisit')['current_page'].ffill()

# Flag de complétion au niveau ligne, pour filtrer facilement ensuite
df_search['completed'] = df_search['idvisit'].isin(completed_visits)

n_search = len(visits_search_landing)
n_search_completed = len(visits_search_landing & completed_visits)
print(f"Visites commençant par la contribution depuis search : {n_search} ({n_search/n_landing_all:.1%} des arrivées directes)")
print(f"  complétées     : {n_search_completed} ({n_search_completed/n_search:.1%})")
print(f"  non complétées : {n_search - n_search_completed} ({(n_search - n_search_completed)/n_search:.1%})")
print(f"Lignes dans df_search : {len(df_search)}")

In [ ]:
IGNORED_CATEGORIES = {'nps'}

events_search = df_search[(df_search['action_type'] == 'event') &
                          ~df_search['action_eventcategory'].isin(IGNORED_CATEGORIES)]

visits_with_interaction = set(events_search['idvisit'].unique())
visits_no_interaction = visits_search_landing - visits_with_interaction

df_search_i = df_search[df_search['idvisit'].isin(visits_with_interaction)].copy()

n_search = len(visits_search_landing)
print(f"Visites search                          : {n_search}")
print(f"  sans interaction intéressante         : {len(visits_no_interaction)} ({len(visits_no_interaction)/n_search:.1%})")
print(f"  avec interaction intéressante         : {len(visits_with_interaction)} ({len(visits_with_interaction)/n_search:.1%})")
print(f"  dont complétées                       : {len(visits_with_interaction & completed_visits)} "
      f"({len(visits_with_interaction & completed_visits)/len(visits_with_interaction):.1%})")

In [ ]:
segments = {
    'sans interaction intéressante': visits_no_interaction,
    'avec interaction intéressante': visits_with_interaction,
}

rows = []
for label, s in segments.items():
    n = len(s)
    n_done = len(s & completed_visits)
    rows.append((label, n, n_done, n - n_done))

completion_by_segment = pd.DataFrame(rows, columns=['segment', 'n_visits', 'n_completed', 'n_not_completed'])
completion_by_segment['pct_visits'] = (completion_by_segment['n_visits'] / n_search * 100).round(1)
completion_by_segment['taux_completion'] = (completion_by_segment['n_completed'] / completion_by_segment['n_visits'] * 100).round(1)
completion_by_segment.loc[len(completion_by_segment)] = [
    'total search', n_search, len(visits_search_landing & completed_visits),
    n_search - len(visits_search_landing & completed_visits), 100.0,
    round(len(visits_search_landing & completed_visits) / n_search * 100, 1)]
completion_by_segment

In [ ]:
CC_SEARCH_CATEGORIES = {'cc_search_type_of_users', 'enterprise_search'}

mask_cc_search = ((df_search_i['action_type'] == 'event') &
                  (df_search_i['action_eventcategory'].isin(CC_SEARCH_CATEGORIES)) &
                  (df_search_i['current_page'] == TARGET_PATH))

visits_cc_search = set(df_search_i.loc[mask_cc_search, 'idvisit'].unique())
visits_no_cc_search = visits_with_interaction - visits_cc_search

df_cc = df_search_i[df_search_i['idvisit'].isin(visits_cc_search)].copy()

n_i = len(visits_with_interaction)
n_cc = len(visits_cc_search)
n_cc_done = len(visits_cc_search & completed_visits)
print(f"Visites avec interaction                    : {n_i}")
print(f"  ayant engagé la recherche de CC           : {n_cc} ({n_cc/n_i:.1%})")
print(f"    complétées                              : {n_cc_done} ({n_cc_done/n_cc:.1%})")
print(f"    non complétées                          : {n_cc - n_cc_done} ({(n_cc - n_cc_done)/n_cc:.1%})")
print(f"  sans recherche de CC                      : {len(visits_no_cc_search)} ({len(visits_no_cc_search)/n_i:.1%})")

In [ ]:
first_choice = (df_cc[mask_cc_search.reindex(df_cc.index).fillna(False)]
                .sort_values(['idvisit', 'actions'])
                .groupby('idvisit')['action_eventaction'].first())

by_choice = pd.DataFrame({
    'n_visits': first_choice.value_counts(),
    'n_completed': first_choice[first_choice.index.isin(completed_visits)].value_counts(),
}).fillna(0).astype(int)
by_choice['pct_visits'] = (by_choice['n_visits'] / n_cc * 100).round(1)
by_choice['taux_completion'] = (by_choice['n_completed'] / by_choice['n_visits'] * 100).round(1)
by_choice

In [ ]:
n_pages_search = (df_search[df_search['action_type'] == 'action']
                  .groupby('idvisit').size())

visits_search_nc = visits_search_landing - completed_visits
visits_search_nc_bounce = {v for v in visits_search_nc if n_pages_search.get(v, 0) <= 1}
visits_search_nc_continue = visits_search_nc - visits_search_nc_bounce

n_nc = len(visits_search_nc)
print(f"Visites search non complétées                     : {n_nc}")
print(f"  sorties sans consulter d'autre page             : {len(visits_search_nc_bounce)} ({len(visits_search_nc_bounce)/n_nc:.1%})")
print(f"  ayant consulté au moins une autre page          : {len(visits_search_nc_continue)} ({len(visits_search_nc_continue)/n_nc:.1%})")

In [ ]:
grp = df_search.sort_values(['idvisit', 'actions']).groupby('idvisit')
time_attrs = grp.agg(visitduration=('visitduration', 'first'),
                     t_first=('action_timestamp', 'min'),
                     t_last=('action_timestamp', 'max'))
time_attrs['visitduration'] = pd.to_numeric(time_attrs['visitduration'], errors='coerce')
time_attrs['observed_s'] = (time_attrs['t_last'] - time_attrs['t_first']).dt.total_seconds()

bins = [-1, 0, 10, 30, 60, 120, 300, 600, float('inf')]
labels = ['0s', '1-10s', '11-30s', '31-60s', '1-2min', '2-5min', '5-10min', '>10min']
time_attrs['dur_bucket'] = pd.cut(time_attrs['visitduration'], bins=bins, labels=labels)
time_attrs['obs_bucket'] = pd.cut(time_attrs['observed_s'], bins=bins, labels=labels)

def segment(v):
    if v in completed_visits: return 'complétée'
    if v in visits_search_nc_bounce: return 'non complétée - sortie sèche'
    return 'non complétée - autre page'
time_attrs['segment'] = [segment(v) for v in time_attrs.index]
time_attrs['interaction'] = time_attrs.index.isin(visits_with_interaction)

In [ ]:
(pd.crosstab(time_attrs['segment'], time_attrs['obs_bucket'], normalize='index') * 100).round(1)

In [ ]:
nps_events = df_search[(df_search['action_type'] == 'event') &
                       (df_search['action_eventcategory'] == 'nps')]
visits_nps = set(nps_events['idvisit'].unique())

n_search = len(visits_search_landing)
print(f"Visites search                 : {n_search}")
print(f"  avec au moins un event nps   : {len(visits_nps)} ({len(visits_nps)/n_search:.1%})")

# Détail des actions nps (affichage du bandeau vs réponse)
(nps_events.groupby(['action_eventaction', 'action_eventname'], dropna=False)['idvisit']
           .nunique().sort_values(ascending=False).head(20))

In [ ]:
visits_nps_nc = visits_nps - completed_visits
nps_time = time_attrs[time_attrs.index.isin(visits_nps_nc)]

print(f"Visites nps non complétées : {len(nps_time)} ({len(nps_time)/len(visits_nps):.1%} des visites nps)")

nps_time[['visitduration', 'observed_s']].describe().round(0)

In [ ]:
dist = pd.DataFrame({
    'visitduration': nps_time['dur_bucket'].value_counts(sort=False),
    'temps_observé': nps_time['obs_bucket'].value_counts(sort=False),
})
(dist / len(nps_time) * 100).round(1)

In [ ]:
nps_time.groupby(['segment', 'interaction']).agg(
    n_visits=('observed_s', 'size'),
    med_visitduration=('visitduration', 'median'),
    med_observed=('observed_s', 'median')).round(0)

In [ ]:
seuils = [10, 30, 60, 120, 300]
rows = []
for s in seuils:
    rows.append((f"< {s}s",
                 round((nps_time['visitduration'] < s).mean() * 100, 1),
                 round((nps_time['observed_s'] < s).mean() * 100, 1)))
pd.DataFrame(rows, columns=['seuil', 'pct_visitduration', 'pct_temps_observé'])

In [ ]:
print(f"{(nps_time['visitduration'] < 10).mean():.1%} des visites nps non complétées ont une visitduration < 10s")
print(f"{(nps_time['observed_s'] < 10).mean():.1%} ont un temps observé < 10s")